In [ ]:
import simpy
import pandas as pd
import numpy as np
import random
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # [수정] 시각화 및 스타일링을 위한 라이브러리
import seaborn as sns               # [추가] 히트맵용 시각화 라이브러리

# 한글 폰트 설정 (히트맵 깨짐 해결)
plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우 한글 폰트 (Mac의 경우 'AppleGothic' 등으로 변경)
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트
STATIONS = {'S01': [3.5, 0.5, 0], 'S02': [11.0, 0.5, 0]} # 작업 스테이션 위치 (x, y, z), CTU가 작업 후 반드시 돌아오는 목적지
CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# [수정] 교차 통로(Cross Aisle) 설정
# 로봇이 다른 통로로 넘어갈 때 랙을 관통하지 않고 앞쪽으로 빠져나와서 이동하는 기준 Y 좌표
CROSS_AISLE_Y = 2.0 

# ==========================================
# [신규] 창고 레이아웃 기반 맨해튼 직교 거리 계산 함수
# ==========================================
def get_manhattan_distance(pos1, pos2):
    x1, y1, z1 = pos1
    x2, y2, z2 = pos2
    
    # 1. 같은 통로(X좌표 동일)일 경우: 단순히 Y축 깊이 차이만 계산
    if x1 == x2:
        xy_dist = abs(y1 - y2)
    # 2. 다른 통로일 경우: 현재 통로에서 교차 통로(Y=2.0)로 직진하여 빠져나온 뒤, 가로(X)로 이동 후 새 통로로 진입
    else:
        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
        
    # Z축(높이) 차이 포함하여 최종 3D 맨해튼 거리 반환 (의사결정 우선순위용)
    return xy_dist + abs(z1 - z2)

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇 정의
    def __init__(self, env, id, warehouse, dispatch_rule): # env: 시뮬레이션 환경, id: CTU 번호, warehouse: 작업정보 접근, dispatch_rule: 작업 할당 알고리즘)
        self.env = env                  # Simpy 시뮬레이션 환경 객체 (현재 시뮬레이션 시간을 알고 있음)
        self.id = id                    # 로봇의 고유 번호 (0, 1, 2...)
        self.warehouse = warehouse      # 로봇이 소속된 창고 객체
        self.dispatch_rule = dispatch_rule # 이 로봇에게 적용될 작업 할당 알고리즘 (nearest, density, load_balance)
        
        # 초기 로봇 위치 설정: 로봇 2대가 서로 다른 스테이션에서 시작하도록 설정.
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] # 0번 CTU는 S01에서 시작, 1번 CTU는 S02에서 시작

        # 총 이동거리 기록
        # 처리 작업 수 기록
        self.total_distance = 0         # 시뮬레이션 동안 로봇이 이동한 총 누적 거리
        self.completed_jobs = 0         # 처리한 총 작업 수 (현재 코드에서는 집계만 하고 사용 안 함)
        self.busy = False               # 로봇이 현재 작업 중인지 여부 (True: 작업중, False: 대기중)
        
        # 핵심: env.process(self.run())
        # CTU의 작업 수행 프로세스를 시뮬레이션에 등록
        # run() 함수가 시간 흐름에 따라 실행
        # 이 객체가 생성되는 순간, self.run() 메서드를 SimPy의 프로세스로 등록
        # 시뮬레이션이 시작(env.run)되면 로봇은 독립적으로 움직임.
        self.process = env.process(self.run())

    # 이동 시간 및 거리 계산
    # 목표 위치까지 이동시간 계산 함수
    def calculate_travel_time(self, target_pos):
        # [수정] 현재 위치(self.pos)와 목표 위치(target_pos) 사이의 거리를 맨해튼 직교 거리로 계산
        x1, y1, z1 = self.pos
        x2, y2, z2 = target_pos
        
        if x1 == x2:
            xy_dist = abs(y1 - y2)
        else:
            xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
            
        dz = abs(z1 - z2)
        
        # 이동 시간 계산 로직:
        # 로봇은 XY 평면 이동과 Z축(리프트) 상승/하강을 '동시에' 할 수 있다고 가정
        # XY 이동 시간과 Z 이동 시간 중 더 오래 걸리는 시간이 최종 소요 시간(max)이 됩니다.
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
        
        # [수정] 총 이동 거리: 맨해튼 궤적 기반 실제 주행 거리 산출
        travel_distance = xy_dist + dz
        
        # 이동시간 + 총 이동거리 반환
        return travel_time, travel_distance

    # CTU의 실제 작업 수행 프로세스
    def run(self): 
        while True: # 시뮬레이션이 끝날 때까지 계속 반복
            # 1. 창고에 처리해야 할 일(job_queue)이 있는지 확인
            # 처리할 작업이 존재하면 '작업중' 상태 변경
            if self.warehouse.job_queue:
                self.busy = True # 작업 시작 상태로 변경
                
                # 2. 작업 할당 규칙에 따라 warehouse에 작업 묶음(batch)을 요청
                # 어떤 작업을 가져올지는 dispatch_rule에 따라 창고가 결정
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    # 3. 경로 최적화 (Greedy TSP)
                    # 할당받은 작업(batch)들 중, 현재 위치 기준 가까운 순서대로 방문 순서를 정렬
                    # [수정] 유클리드 대신 맨해튼 거리 산출 함수 적용
                    batch.sort(key=lambda j: get_manhattan_distance(self.pos, j['pos']))
                    
                    # 4. 각 상품 위치로 이동하며 피킹 수행
                    for job in batch:
                        # 목표 위치까지 걸리는 시간(t)과 거리(d) 계산
                        t, d = self.calculate_travel_time(job['pos'])
                        
                        # [추가] 이동 로그 기록
                        self.warehouse.move_log.append({
                            'time': self.env.now,
                            'ctu_id': self.id,
                            'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2],
                            'to_x': job['pos'][0], 'to_y': job['pos'][1], 'to_z': job['pos'][2],
                            'item_id': job['item_id'],
                            'travel_time': t, 'travel_distance': d,
                            'dispatch_rule': self.dispatch_rule,
                        })
                        
                        # "yield env.timeout(t)"
                        # 로봇이 이동하는 시간(t초) 동안 프로세스를 멈추고 대기
                        # 실제 이동시간만큼 시뮬레이션 시간 흐름 발생
                        yield self.env.timeout(t)
                        
                        self.pos = job['pos']     # 로봇 위치를 목표 지점으로 업데이트
                        self.total_distance += d  # 이동 거리 누적
                        self.completed_jobs += 1
                        
                    # 5. 모든 상품 피킹 완료 후, 가까운 스테이션으로 복귀
                    # [수정] 작업이 끝난 통로(X좌표) 위치에 따라 복귀할 지정 스테이션 결정
                    last_job_x = self.pos[0]
                    # X좌표가 6.0 이하(A01, A02, A03)면 S01, 아니면(A04, A05, A06) S02로 고정 복귀
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02']
                    
                    # 선택된 스테이션까지 이동시간 및 거리 계산
                    t, d = self.calculate_travel_time(target_station)
                    
                    # [추가] 복귀 이동 로그
                    self.warehouse.move_log.append({
                        'time': self.env.now,
                        'ctu_id': self.id,
                        'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2],
                        'to_x': target_station[0], 'to_y': target_station[1], 'to_z': target_station[2],
                        'item_id': 'RETURN',
                        'travel_time': t, 'travel_distance': d,
                        'dispatch_rule': self.dispatch_rule,
                    })
                    
                    yield self.env.timeout(t) # 이동 시간만큼 시뮬레이션 시간 흐름 
                    self.pos = target_station           # 위치 업데이트
                    self.total_distance += d  # 이동거리 누적
                
                self.busy = False # 작업 완료 후 상태를 대기중(idle)으로 변경
                
            else:
                # 6. 할 일이 없으면 1초 동안 대기했다가 다시 큐를 확인 (Busy Waiting 방지)
                yield self.env.timeout(1)


# ==========================================
# 창고 시스템 클래스 정의
# 재고 위치 및 작업 큐 관리 역할
# ==========================================
class Warehouse:
    def __init__(self, env, inventory, plan_df):
        self.env = env              # 시뮬레이션 환경 저장
        self.inventory = inventory # {부품코드: [x, y, z]} 형태의 딕셔너리 (부품별 저장 위치 정보, Slotting 결과)
        self.plan_df = plan_df      # 해당 날짜에 처리해야 할 출고 계획 데이터프레임
        self.job_queue = []         # 실시간으로 들어오는 주문들이 쌓이는 대기열, 아직 처리되지 않은 Picking 작업 큐
        
        # [추가] 분석용 로그 리스트
        self.dispatch_log = []      # 배치 구성 로그
        self.move_log = []          # CTU 이동 로그

    # dispatching rule에 따라 CTU에게 할당할 작업 배치 생성 
    def apply_dispatching(self, ctu, rule):
        # 대기 중인 작업이 없으면 빈 리스트 반환
        if not self.job_queue: return []

        # ==========================================
        # 1. Nearest-seeded batch
        # ==========================================
        if rule == 'nearest':
            # 현재 위치에서 가장 물리적 거리가 가까운 작업 순서로 정렬 (Nearest seed job 하나 선택)
            # [수정] 맨해튼 직교 거리 기준 적용
            self.job_queue.sort(key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            
            # 첫 번째 작업(Nearest seed)을 기준으로 삼음
            batch = [self.job_queue.pop(0)] 
            
            if self.job_queue:
                # Seed 주변에서 경로 상 가까운 job을 추가하기 위해 다시 정렬 
                self.job_queue.sort(key=lambda j: get_manhattan_distance(batch[0]['pos'], j['pos']))
                
                # 한 번에 최대 7개 box 로드 가능
                extra_count = min(len(self.job_queue), TOTE_CAPACITY - 1)
                
                # Q개가 될 때까지 추가하여 할당 및 큐에서 제거 
                batch.extend(self.job_queue[:extra_count])
                del self.job_queue[:extra_count]
                
            # [추가] 배치 로그 기록
            self.dispatch_log.append({
                'time': ctu.env.now,
                'ctu_id': ctu.id,
                'rule': 'nearest',
                'seed_item': batch[0]['item_id'],
                'batch_items': [j['item_id'] for j in batch],
                'batch_size': len(batch),
            })
            return batch

        # ==========================================
        # 2. Density-seeded batch 
        # ==========================================
        elif rule == 'density':
            # [수정] 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            candidate_seeds = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))[:K]
            
            best_seed = None
            max_density = -1
            r_y = 2.65 # [수정] 1개 베이 길이 (m) 적용 (업라이트 및 박스 여유 공간 50mm 포함된 추정 길이) 
            
            # Seed 마다 밀집도 계산
            for seed in candidate_seeds:
                density = 0
                for job in self.job_queue:
                    # 1. 동일 복도(Aisle)에 있는지 확인하여 랙 관통 탐색을 철저히 방지 (폭 1200mm 제약 고려) 
                    if job['pos'][0] == seed['pos'][0]:
                        # 2. 같은 복도라면, Seed 주변 반경 r안에 대기 job이 몇 개 있는지 세서 밀집도 정량화 
                        dy = abs(job['pos'][1] - seed['pos'][1])
                        if dy <= r_y:
                            density += 1
                
                # 가장 밀집도가 큰 seed 갱신 
                if density > max_density:
                    max_density = density
                    best_seed = seed
                    
            # 가장 밀집도가 큰 seed를 선택하여 nearest로 batch 만들어 할당 
            # 선택된 best_seed를 기준으로 전체 대기열을 가까운 순으로 정렬
            self.job_queue.sort(key=lambda j: get_manhattan_distance(best_seed['pos'], j['pos']))
            
            # CTU가 한 번에 로드 가능한 최대 7개 할당 
            batch_size = min(len(self.job_queue), TOTE_CAPACITY)
            best_batch = self.job_queue[:batch_size]
            
            # 할당된 작업들은 큐에서 일괄 제거
            del self.job_queue[:batch_size]
            
            # [추가] 밀집도 로그 포함
            self.dispatch_log.append({
                'time': ctu.env.now,
                'ctu_id': ctu.id,
                'rule': 'density',
                'seed_item': best_seed['item_id'],
                'batch_items': [j['item_id'] for j in best_batch],
                'batch_size': len(best_batch),
                'local_density': max_density,
            })
            return best_batch

        # ==========================================
        # 3. Load-balance batch 
        # ==========================================
        elif rule == 'load_balance':
            # 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            
            # [수정] 맨해튼 거리 기준 상위 10개 추출
            nearest_jobs = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            candidate_seeds = nearest_jobs[:K]
            
            best_batch = []
            min_eta = float('inf') 
            best_seed_item = None
            
            # Seed 마다 nearest로 batch 만들기
            for seed in candidate_seeds:
                temp_queue = sorted(self.job_queue, key=lambda j: get_manhattan_distance(seed['pos'], j['pos']))
                
                # 최대 7개 box 로드 
                temp_batch = temp_queue[:TOTE_CAPACITY]
                
                # Batch 별 예상종료시간(ETA, Estimated Time of Arrival) 계산 (batch 내부 이동 시간)
                eta = 0
                current_pos = ctu.pos 
                
                for job in temp_batch:
                    # [수정] ETA 계산 시에도 로봇의 교차 통로 이동 논리 적용
                    x1, y1, z1 = current_pos
                    x2, y2, z2 = job['pos']
                    
                    if x1 == x2:
                        xy_dist = abs(y1 - y2)
                    else:
                        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
                        
                    dz = abs(z1 - z2)
                    
                    # 로봇 이동 시간 산출 (XY 평면 속도 및 리프팅 속도 고려) 
                    time_to_job = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
                    eta += time_to_job
                    current_pos = job['pos'] 
                
                # ETA가 가장 작은 batch 선택하여 할당 
                if eta < min_eta:
                    min_eta = eta
                    best_batch = temp_batch
                    best_seed_item = seed['item_id']
            
            # 최적 배치를 찾았으므로 전체 대기열(job_queue)에서 제거 후 반환
            for job in best_batch:
                self.job_queue.remove(job)
                
            # [추가] ETA 로그 포함
            self.dispatch_log.append({
                'time': ctu.env.now,
                'ctu_id': ctu.id,
                'rule': 'load_balance',
                'seed_item': best_seed_item,
                'batch_items': [j['item_id'] for j in best_batch],
                'batch_size': len(best_batch),
                'eta': min_eta,
            })
            return best_batch


# ==========================================
# Slotting Rule
# ==========================================
def generate_fixed_inventory(rule_type, part_list, full_plan_df):
    inventory = {} # 부품 위치 저장용 딕셔너리 생성
    all_parts = sorted(part_list) # 모든 부품 코드 정렬
    
    # [수정] 랜덤 고정 시드 적용 -> 결과 재현
    random.seed(1234)
    np.random.seed(1234)
    
    # 공통: 부품별 출고 빈도 계산
    freq = full_plan_df['부품코드'].value_counts().to_dict()
    sorted_by_freq = sorted(all_parts, key=lambda x: freq.get(x, 0), reverse=True)
    n = len(all_parts)

    # 1. Random Slotting
    if rule_type == 'random':
        for part in all_parts:
            # 통로(X), 랙(Y), 선반(Z) 모두 random 선택
            inventory[part] = [random.choice(AISLE_X), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]

    # 2. ABC Slotting 
    elif rule_type == 'abc':
        # 상위 20% A, 다음 30% B, 나머지 C
        nA = max(1, int(n * 0.2))
        nB = max(1, int(n * 0.3))
        A_parts = sorted_by_freq[:nA]
        B_parts = sorted_by_freq[nA:nA+nB]
        C_parts = sorted_by_freq[nA+nB:]

        # A: 스테이션 가까운 AISLE (3.6, 6.0, 8.4) / 낮은 Y / 낮은 Z
        near_aisles = AISLE_X[1:4]  # A02, A03, A04
        far_aisles = [AISLE_X[0], AISLE_X[-1]]  # A01, A06
        
        def place_group(parts, aisles, y_range, z_indices):
            for i, part in enumerate(parts):
                aisle = aisles[i % len(aisles)]
                y = random.uniform(*y_range)
                z_idx = z_indices[i % len(z_indices)]
                z = SHELF_HEIGHTS[z_idx]
                inventory[part] = [aisle, round(y, 2), z]

        # A: 입구쪽 / 하층 (Y<9.5, Z=0.0~1.0)
        place_group(A_parts, near_aisles, (RACK_START_Y, 9.5), [0, 1])
        # B: 중간 위치 / 중간층 (Y=7~12, Z=1.0~2.0)
        place_group(B_parts, AISLE_X, (7.0, 12.0), [2, 3])
        # C: 깊은 위치 / 상층 (Y>10.5, Z=2.0~3.0)
        place_group(C_parts, far_aisles + AISLE_X, (10.5, RACK_END_Y), [4, 5, 6])

    # 3. Characteristic Slotting
    elif rule_type == 'characteristic':
        group_u = [p for p in all_parts if str(p).startswith('U')]
        group_c = [p for p in all_parts if str(p).startswith('C')]
        others = [p for p in all_parts if p not in group_u + group_c]

        def place_group(parts, aisles):
            for i, part in enumerate(parts):
                aisle = aisles[i % len(aisles)]
                y = round(random.uniform(RACK_START_Y, RACK_END_Y), 2)
                z = random.choice(SHELF_HEIGHTS)
                inventory[part] = [aisle, y, z]

        # U는 좌측 Aisle, C는 우측 Aisle, 나머지는 가운데
        left_aisles = AISLE_X[:3]      # A01~A03
        right_aisles = AISLE_X[3:]     # A04~A06
        mid_aisles = AISLE_X[1:5]      # A02~A05
        
        place_group(group_u, left_aisles)
        place_group(group_c, right_aisles)
        place_group(others, mid_aisles)

    # 4. Kitting Slotting (같이 나가는 건 같이 둠)
    elif rule_type == 'kitting':
        # 주문 번호(No.)별로 묶어서 같이 주문된 부품 리스트를 추출
        order_groups = full_plan_df.groupby('No.')['부품코드'].apply(list)
        pairs = []
        for items in order_groups:
            # 2개 이상 주문된 건에 대해 모든 쌍(Pair)을 구함
            items = list(set(items))
            if len(items) > 1:
                pairs.extend(list(combinations(sorted(items), 2)))
        
        # 부품쌍의 등장 빈도 계산(많이 같이 등장한 조합 찾기)
        pair_counts = Counter(pairs)
        
        # 연관성 높은 부품부터 순서대로 리스트 생성
        related_parts = []
        for (p1, p2), count in pair_counts.most_common():
            if p1 in all_parts and p1 not in related_parts: 
                related_parts.append(p1)
            if p2 in all_parts and p2 not in related_parts: 
                related_parts.append(p2)
        # 연관성 데이터가 없는 나머지 부품들도 뒤에 추가
        for p in all_parts:
            if p not in related_parts: 
                related_parts.append(p)

        # 연관된 것끼리 같은 통로에 모으는 로직
        # 통로별 배치 수 계산
        parts_per_aisle = len(all_parts) // len(AISLE_X) + 1
        # 위치 배치
        for i, part in enumerate(related_parts):
            aisle_idx = i // parts_per_aisle
            aisle = AISLE_X[min(aisle_idx, len(AISLE_X)-1)]
            
            # 같은 통로 내에서도 인접하게 배치 (y 좌표 1m 이내)
            y_ratio = (i % parts_per_aisle) / parts_per_aisle
            y_pos = RACK_START_Y + y_ratio * (RACK_END_Y - RACK_START_Y)
            
            inventory[part] = [aisle, round(y_pos, 2), random.choice(SHELF_HEIGHTS)]
    
    return inventory


# ==========================================
# 하루 단위 시뮬레이션 실행
# 특정 날짜, 특정 규칙을 적용하여 하루치 작업을 시뮬레이션
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan):
    env = simpy.Environment() # 시뮬레이션 환경 생성
    warehouse = Warehouse(env, inventory, date_plan) # 창고 객체 생성
    
    # CTU 2대 생성 (각각 독립적인 프로세스로 동작 시작)
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)]
    
    # Generator
    def job_gen(env, wh):
        # 출고계획 데이터(plan_df)를 주문번호(No.)별로 그룹화
        for _, group in wh.plan_df.groupby('No.'):
            # 주문과 주문 사이에 2~5초 랜덤 간격 발생 (주문 유입 시뮬레이션)
            yield env.timeout(random.uniform(2, 5))
            
            # 해당 주문에 포함된 모든 부품을 큐에 추가
            for _, row in group.iterrows():
                p_id = row['부품코드']
                if p_id in wh.inventory:
                    # 큐에 작업 추가 (부품ID, 위치, 발생시간)
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now})

    # 종료 조건 확인용 모니터링 함수
    def monitor(env, warehouse, gen_proc, ctus):
        # 1. 모든 주문이 생성될 때까지 대기
        yield gen_proc
        
        # 2. 주문 생성이 끝났어도, 큐에 잔여 작업이 있거나 하나의 CTU라도 작업 중(busy=True)이면 반복 실행
        while warehouse.job_queue or any(c.busy for c in ctus):
            yield env.timeout(1) # 1초 단위로 시스템 상태를 계속 확인하도록 시간 진행

    # 주문 발생 프로세스 실행
    gen_proc = env.process(job_gen(env, warehouse))

    # 시뮬레이션 실행 시작
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus)))
    
    # 시뮬레이션 종료 후 결과 반환
    return {
        'Date': date, 
        'Time': round(env.now, 1), 
        'Distance': round(sum(c.total_distance for c in ctus), 1),
        'Move_log': pd.DataFrame(warehouse.move_log),
        'Dispatch_log': pd.DataFrame(warehouse.dispatch_log),
    }


# ==========================================
# [수정] 히트맵 시각화 함수 (전체 조합 표시용)
# ==========================================
def visualize_heatmap(move_df):
    """Aisle별 피킹 히트맵 시각화"""
    if move_df.empty:
        print("move_log 데이터가 없습니다.")
        return

    # 'RETURN' 이동을 제외하고, Y축에 표시할 텍스트 조합(Slotting × Dispatching) 컬럼 생성
    target_df = move_df[move_df['item_id'] != 'RETURN'].copy()
    target_df['Rule_Combo'] = target_df['Slotting'] + " × " + target_df['Dispatching']

    # Rule_Combo(행) x to_x(열) 기준으로 피킹 횟수 집계
    aisle_picks = target_df.groupby(['Rule_Combo', 'to_x']).size().unstack(fill_value=0)

    # 전체 조합(12개)을 넉넉히 보여주기 위해 세로 길이를 조정
    plt.figure(figsize=(12, 10))

    # Aisle 라벨을 보기 좋게 변환 (예: 첫 번째 열 → A01, 두 번째 열 → A02 ...)
    aisle_labels = [f"A{i+1:02d}" for i in range(len(aisle_picks.columns))]

    sns.heatmap(
        aisle_picks,  # 행이 조합, 열이 통로로 자동 매핑됨
        annot=True,
        fmt='d',
        cmap='YlOrRd',
        cbar_kws={'label': '피킹 횟수'},
        xticklabels=aisle_labels
    )

    plt.title('전체 조합 Aisle별 피킹 히트맵 (Slotting × Dispatching)', fontsize=16, fontweight='bold')
    plt.xlabel('Aisle (통로 번호)')
    plt.ylabel('알고리즘 조합 (Slotting × Dispatching)')
    plt.tight_layout()
    plt.show()


# ==========================================
# [추가] 로그 분석 함수
# ==========================================
def analyze_logs(move_df, dispatch_df, title_prefix=""):
    """로그 데이터 간단 분석"""
    print(f"\n{'='*60}")
    print(f" [{title_prefix}] 분석 결과")
    print(f"{'='*60}")
    
    if not move_df.empty:
        print(f"\n CTU별 총 이동거리:")
        ctu_dist = move_df.groupby('ctu_id')['travel_distance'].sum()
        print(ctu_dist.round(1))
        
        print(f"\n 평균 배치당 이동거리:")
        avg_dist_per_batch = move_df[move_df['item_id'] != 'RETURN'].groupby(
            ['dispatch_rule', 'ctu_id']
        )['travel_distance'].mean()
        print(avg_dist_per_batch.round(1))
    
    if not dispatch_df.empty:
        print(f"\n 배치 크기 통계:")
        batch_stats = dispatch_df.groupby('rule')['batch_size'].agg(['mean', 'std', 'min', 'max']).round(1)
        print(batch_stats)


# ==========================================
# main 함수
# ==========================================
def main():
    try:
        # outbound_plan.csv 파일을 읽어 전체 출고계획 데이터를 DataFrame으로 저장
        plan_df_full = pd.read_csv('outbound_plan.csv')
        # BOM 데이터 생성
        part_list = plan_df_full['부품코드'].unique()
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    # 출고계획 데이터에 존재하는 날짜 목록 추출 및 정렬
    unique_dates = sorted(plan_df_full['생산일자'].unique())
    
    # 적용한 Slotting 알고리즘 목록
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting'] 
    dispatch_rules = ['nearest', 'density', 'load_balance']     

    all_results_summary = []
    move_logs_all = []
    dispatch_logs_all = []

    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다.")
    
    # 모든 경우의 수(Grid Search) 실행
    # 1. Slotting rule 변경
    for s_rule in slotting_rules:
        fixed_inventory = generate_fixed_inventory(s_rule, part_list, plan_df_full)
        
        # 2. Dispatching rule 변경
        for d_rule in dispatch_rules:
            # 3. 날짜별 시뮬레이션 실행
            for date in unique_dates:
                # 해당 날짜 데이터만 필터링
                date_plan = plan_df_full[plan_df_full['생산일자'] == date].copy()
                
                # 시뮬레이션 1회 실행
                res = run_simulation(date, fixed_inventory, d_rule, date_plan)
                
                # 결과 딕셔너리에 사용된 Slotting + Dispatching Rule 정보 추가
                all_results_summary.append({
                    'Date': res['Date'],
                    'Time': res['Time'],
                    'Distance': res['Distance'],
                    'Slotting': s_rule,
                    'Dispatching': d_rule,
                })
                
                # [추가] 로그에 룰/날짜 정보 추가
                move_tmp = res['Move_log'].copy()
                move_tmp['Slotting'] = s_rule
                move_tmp['Dispatching'] = d_rule
                move_tmp['Date'] = date
                move_logs_all.append(move_tmp)
                
                disp_tmp = res['Dispatch_log'].copy()
                disp_tmp['Slotting'] = s_rule
                disp_tmp['Dispatching'] = d_rule
                disp_tmp['Date'] = date
                dispatch_logs_all.append(disp_tmp)

    # 결과 분석 및 출력
    df_res = pd.DataFrame(all_results_summary) # 모든 결과를 DataFrame 형태로 변환
    
    # 피벗 테이블 생성: 행 = Rule 조합, 열 = 날짜, 값 = 소요시간
    pivoted_time = df_res.pivot_table(index=['Slotting', 'Dispatching'], columns='Date', values='Time')
    
    # Rule 조합의 날짜별 총 소요시간 합계 계산
    pivoted_time['Total'] = pivoted_time.sum(axis=1)
    
    # 시간이 적게 걸린 순서로 정렬
    pivoted_time = pivoted_time.sort_values(by='Total')

    # =========================
    # --- 결과 표 & 그래프 ---
    # =========================
    print("\n" + "="*95)
    print(" [최종 시뮬레이션 결과] 알고리즘 조합별 일자별 런타임 요약")
    print("-" * 95)
    
    try:
        from IPython.display import display
        styled = pivoted_time.style.background_gradient(cmap='YlGnBu', subset=pivoted_time.columns[:-1]) \
            .highlight_min(subset=['Total'], color='lightgreen').format("{:.1f}")
        display(styled)
    except:
        print(pivoted_time.to_string())  # 텍스트 출력 fallback
        
    print("="*95)
    
    # 최적 알고리즘 조합 추출 및 출력
    best = pivoted_time.index[0]
    print(f" 최적 알고리즘 조합: Slotting[{best[0]}] + Dispatching[{best[1]}]")
    print(f"  최적 총 소요시간: {pivoted_time.loc[best, 'Total']:.1f}초")

    # 전체 로그 병합
    move_all_df = pd.concat(move_logs_all, ignore_index=True) if move_logs_all else pd.DataFrame()
    dispatch_all_df = pd.concat(dispatch_logs_all, ignore_index=True) if dispatch_logs_all else pd.DataFrame()
    
    # 로그 분석 출력
    analyze_logs(move_all_df, dispatch_all_df, title_prefix="전체 결과")
    
    # [수정] 히트맵 시각화 (전체 조합 모두 표시)
    print("\n 전체 알고리즘 조합에 대한 Aisle별 피킹 히트맵을 생성합니다...")
    visualize_heatmap(move_all_df)


# 현재 파일이 직접 실행될 때만 main() 함수 호출
if __name__ == "__main__":
    main()

In [ ]:
import simpy
import pandas as pd
import numpy as np
import random
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # [수정] 시각화 및 스타일링을 위한 라이브러리
import seaborn as sns               # [추가] 히트맵용 시각화 라이브러리

# 한글 폰트 설정 (히트맵 깨짐 해결)
plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우 한글 폰트 (Mac의 경우 'AppleGothic' 등으로 변경)
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트
STATIONS = {'S01': [3.5, 0.5, 0], 'S02': [11.0, 0.5, 0]} # 작업 스테이션 위치 (x, y, z), CTU가 작업 후 반드시 돌아오는 목적지
CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# [수정] 교차 통로(Cross Aisle) 설정
# 로봇이 다른 통로로 넘어갈 때 랙을 관통하지 않고 앞쪽으로 빠져나와서 이동하는 기준 Y 좌표
CROSS_AISLE_Y = 2.0 

# ==========================================
# [신규] 창고 레이아웃 기반 맨해튼 직교 거리 계산 함수
# ==========================================
def get_manhattan_distance(pos1, pos2):
    x1, y1, z1 = pos1
    x2, y2, z2 = pos2
    
    # 1. 같은 통로(X좌표 동일)일 경우: 단순히 Y축 깊이 차이만 계산
    if x1 == x2:
        xy_dist = abs(y1 - y2)
    # 2. 다른 통로일 경우: 현재 통로에서 교차 통로(Y=2.0)로 직진하여 빠져나온 뒤, 가로(X)로 이동 후 새 통로로 진입
    else:
        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
        
    # Z축(높이) 차이 포함하여 최종 3D 맨해튼 거리 반환 (의사결정 우선순위용)
    return xy_dist + abs(z1 - z2)

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇 정의
    def __init__(self, env, id, warehouse, dispatch_rule): # env: 시뮬레이션 환경, id: CTU 번호, warehouse: 작업정보 접근, dispatch_rule: 작업 할당 알고리즘)
        self.env = env                  # Simpy 시뮬레이션 환경 객체 (현재 시뮬레이션 시간을 알고 있음)
        self.id = id                    # 로봇의 고유 번호 (0, 1, 2...)
        self.warehouse = warehouse      # 로봇이 소속된 창고 객체
        self.dispatch_rule = dispatch_rule # 이 로봇에게 적용될 작업 할당 알고리즘 (nearest, density, load_balance)
        
        # 초기 로봇 위치 설정: 로봇 2대가 서로 다른 스테이션에서 시작하도록 설정.
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] # 0번 CTU는 S01에서 시작, 1번 CTU는 S02에서 시작

        # 총 이동거리 기록
        # 처리 작업 수 기록
        self.total_distance = 0         # 시뮬레이션 동안 로봇이 이동한 총 누적 거리
        self.completed_jobs = 0         # 처리한 총 작업 수 (현재 코드에서는 집계만 하고 사용 안 함)
        self.busy = False               # 로봇이 현재 작업 중인지 여부 (True: 작업중, False: 대기중)
        
        # 핵심: env.process(self.run())
        # CTU의 작업 수행 프로세스를 시뮬레이션에 등록
        # run() 함수가 시간 흐름에 따라 실행
        # 이 객체가 생성되는 순간, self.run() 메서드를 SimPy의 프로세스로 등록
        # 시뮬레이션이 시작(env.run)되면 로봇은 독립적으로 움직임.
        self.process = env.process(self.run())

    # 이동 시간 및 거리 계산
    # 목표 위치까지 이동시간 계산 함수
    def calculate_travel_time(self, target_pos):
        # [수정] 현재 위치(self.pos)와 목표 위치(target_pos) 사이의 거리를 맨해튼 직교 거리로 계산
        x1, y1, z1 = self.pos
        x2, y2, z2 = target_pos
        
        if x1 == x2:
            xy_dist = abs(y1 - y2)
        else:
            xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
            
        dz = abs(z1 - z2)
        
        # 이동 시간 계산 로직:
        # 로봇은 XY 평면 이동과 Z축(리프트) 상승/하강을 '동시에' 할 수 있다고 가정
        # XY 이동 시간과 Z 이동 시간 중 더 오래 걸리는 시간이 최종 소요 시간(max)이 됩니다.
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
        
        # [수정] 총 이동 거리: 맨해튼 궤적 기반 실제 주행 거리 산출
        travel_distance = xy_dist + dz
        
        # 이동시간 + 총 이동거리 반환
        return travel_time, travel_distance

    # CTU의 실제 작업 수행 프로세스
    def run(self): 
        while True: # 시뮬레이션이 끝날 때까지 계속 반복
            # 1. 창고에 처리해야 할 일(job_queue)이 있는지 확인
            # 처리할 작업이 존재하면 '작업중' 상태 변경
            if self.warehouse.job_queue:
                self.busy = True # 작업 시작 상태로 변경
                
                # 2. 작업 할당 규칙에 따라 warehouse에 작업 묶음(batch)을 요청
                # 어떤 작업을 가져올지는 dispatch_rule에 따라 창고가 결정
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    # 3. 경로 최적화 (Greedy TSP)
                    # 할당받은 작업(batch)들 중, 현재 위치 기준 가까운 순서대로 방문 순서를 정렬
                    # [수정] 유클리드 대신 맨해튼 거리 산출 함수 적용
                    batch.sort(key=lambda j: get_manhattan_distance(self.pos, j['pos']))
                    
                    # 4. 각 상품 위치로 이동하며 피킹 수행
                    for job in batch:
                        # 목표 위치까지 걸리는 시간(t)과 거리(d) 계산
                        t, d = self.calculate_travel_time(job['pos'])
                        
                        # [추가] 이동 로그 기록
                        self.warehouse.move_log.append({
                            'time': self.env.now,
                            'ctu_id': self.id,
                            'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2],
                            'to_x': job['pos'][0], 'to_y': job['pos'][1], 'to_z': job['pos'][2],
                            'item_id': job['item_id'],
                            'travel_time': t, 'travel_distance': d,
                            'dispatch_rule': self.dispatch_rule,
                        })
                        
                        # "yield env.timeout(t)"
                        # 로봇이 이동하는 시간(t초) 동안 프로세스를 멈추고 대기
                        # 실제 이동시간만큼 시뮬레이션 시간 흐름 발생
                        yield self.env.timeout(t)
                        
                        self.pos = job['pos']     # 로봇 위치를 목표 지점으로 업데이트
                        self.total_distance += d  # 이동 거리 누적
                        self.completed_jobs += 1
                        
                    # 5. 모든 상품 피킹 완료 후, 가까운 스테이션으로 복귀
                    # [수정] 작업이 끝난 통로(X좌표) 위치에 따라 복귀할 지정 스테이션 결정
                    last_job_x = self.pos[0]
                    # X좌표가 6.0 이하(A01, A02, A03)면 S01, 아니면(A04, A05, A06) S02로 고정 복귀
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02']
                    
                    # 선택된 스테이션까지 이동시간 및 거리 계산
                    t, d = self.calculate_travel_time(target_station)
                    
                    # [추가] 복귀 이동 로그
                    self.warehouse.move_log.append({
                        'time': self.env.now,
                        'ctu_id': self.id,
                        'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2],
                        'to_x': target_station[0], 'to_y': target_station[1], 'to_z': target_station[2],
                        'item_id': 'RETURN',
                        'travel_time': t, 'travel_distance': d,
                        'dispatch_rule': self.dispatch_rule,
                    })
                    
                    yield self.env.timeout(t) # 이동 시간만큼 시뮬레이션 시간 흐름 
                    self.pos = target_station           # 위치 업데이트
                    self.total_distance += d  # 이동거리 누적
                
                self.busy = False # 작업 완료 후 상태를 대기중(idle)으로 변경
                
            else:
                # 6. 할 일이 없으면 1초 동안 대기했다가 다시 큐를 확인 (Busy Waiting 방지)
                yield self.env.timeout(1)


# ==========================================
# 창고 시스템 클래스 정의
# 재고 위치 및 작업 큐 관리 역할
# ==========================================
class Warehouse:
    def __init__(self, env, inventory, plan_df):
        self.env = env              # 시뮬레이션 환경 저장
        self.inventory = inventory # {부품코드: [x, y, z]} 형태의 딕셔너리 (부품별 저장 위치 정보, Slotting 결과)
        self.plan_df = plan_df      # 해당 날짜에 처리해야 할 출고 계획 데이터프레임
        self.job_queue = []         # 실시간으로 들어오는 주문들이 쌓이는 대기열, 아직 처리되지 않은 Picking 작업 큐
        
        # [추가] 분석용 로그 리스트
        self.dispatch_log = []      # 배치 구성 로그
        self.move_log = []          # CTU 이동 로그

    # dispatching rule에 따라 CTU에게 할당할 작업 배치 생성 
    def apply_dispatching(self, ctu, rule):
        # 대기 중인 작업이 없으면 빈 리스트 반환
        if not self.job_queue: return []

        # ==========================================
        # 1. Nearest-seeded batch
        # ==========================================
        if rule == 'nearest':
            # 현재 위치에서 가장 물리적 거리가 가까운 작업 순서로 정렬 (Nearest seed job 하나 선택)
            # [수정] 맨해튼 직교 거리 기준 적용
            self.job_queue.sort(key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            
            # 첫 번째 작업(Nearest seed)을 기준으로 삼음
            batch = [self.job_queue.pop(0)] 
            
            if self.job_queue:
                # Seed 주변에서 경로 상 가까운 job을 추가하기 위해 다시 정렬 
                self.job_queue.sort(key=lambda j: get_manhattan_distance(batch[0]['pos'], j['pos']))
                
                # 한 번에 최대 7개 box 로드 가능
                extra_count = min(len(self.job_queue), TOTE_CAPACITY - 1)
                
                # Q개가 될 때까지 추가하여 할당 및 큐에서 제거 
                batch.extend(self.job_queue[:extra_count])
                del self.job_queue[:extra_count]
                
            # [추가] 배치 로그 기록
            self.dispatch_log.append({
                'time': ctu.env.now,
                'ctu_id': ctu.id,
                'rule': 'nearest',
                'seed_item': batch[0]['item_id'],
                'batch_items': [j['item_id'] for j in batch],
                'batch_size': len(batch),
            })
            return batch

        # ==========================================
        # 2. Density-seeded batch 
        # ==========================================
        elif rule == 'density':
            # [수정] 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            candidate_seeds = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))[:K]
            
            best_seed = None
            max_density = -1
            r_y = 2.65 # [수정] 1개 베이 길이 (m) 적용 (업라이트 및 박스 여유 공간 50mm 포함된 추정 길이) 
            
            # Seed 마다 밀집도 계산
            for seed in candidate_seeds:
                density = 0
                for job in self.job_queue:
                    # 1. 동일 복도(Aisle)에 있는지 확인하여 랙 관통 탐색을 철저히 방지 (폭 1200mm 제약 고려) 
                    if job['pos'][0] == seed['pos'][0]:
                        # 2. 같은 복도라면, Seed 주변 반경 r안에 대기 job이 몇 개 있는지 세서 밀집도 정량화 
                        dy = abs(job['pos'][1] - seed['pos'][1])
                        if dy <= r_y:
                            density += 1
                
                # 가장 밀집도가 큰 seed 갱신 
                if density > max_density:
                    max_density = density
                    best_seed = seed
                    
            # 가장 밀집도가 큰 seed를 선택하여 nearest로 batch 만들어 할당 
            # 선택된 best_seed를 기준으로 전체 대기열을 가까운 순으로 정렬
            self.job_queue.sort(key=lambda j: get_manhattan_distance(best_seed['pos'], j['pos']))
            
            # CTU가 한 번에 로드 가능한 최대 7개 할당 
            batch_size = min(len(self.job_queue), TOTE_CAPACITY)
            best_batch = self.job_queue[:batch_size]
            
            # 할당된 작업들은 큐에서 일괄 제거
            del self.job_queue[:batch_size]
            
            # [추가] 밀집도 로그 포함
            self.dispatch_log.append({
                'time': ctu.env.now,
                'ctu_id': ctu.id,
                'rule': 'density',
                'seed_item': best_seed['item_id'],
                'batch_items': [j['item_id'] for j in best_batch],
                'batch_size': len(best_batch),
                'local_density': max_density,
            })
            return best_batch

        # ==========================================
        # 3. Load-balance batch 
        # ==========================================
        elif rule == 'load_balance':
            # 후보 seed 확인: 아직 안 끝난 job 중에서 MLTU 기준으로 가까운 순 Top-K개 확인 (K = 10 정도) 
            K = min(10, len(self.job_queue)) 
            
            # [수정] 맨해튼 거리 기준 상위 10개 추출
            nearest_jobs = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            candidate_seeds = nearest_jobs[:K]
            
            best_batch = []
            min_eta = float('inf') 
            best_seed_item = None
            
            # Seed 마다 nearest로 batch 만들기
            for seed in candidate_seeds:
                temp_queue = sorted(self.job_queue, key=lambda j: get_manhattan_distance(seed['pos'], j['pos']))
                
                # 최대 7개 box 로드 
                temp_batch = temp_queue[:TOTE_CAPACITY]
                
                # Batch 별 예상종료시간(ETA, Estimated Time of Arrival) 계산 (batch 내부 이동 시간)
                eta = 0
                current_pos = ctu.pos 
                
                for job in temp_batch:
                    # [수정] ETA 계산 시에도 로봇의 교차 통로 이동 논리 적용
                    x1, y1, z1 = current_pos
                    x2, y2, z2 = job['pos']
                    
                    if x1 == x2:
                        xy_dist = abs(y1 - y2)
                    else:
                        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
                        
                    dz = abs(z1 - z2)
                    
                    # 로봇 이동 시간 산출 (XY 평면 속도 및 리프팅 속도 고려) 
                    time_to_job = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
                    eta += time_to_job
                    current_pos = job['pos'] 
                
                # ETA가 가장 작은 batch 선택하여 할당 
                if eta < min_eta:
                    min_eta = eta
                    best_batch = temp_batch
                    best_seed_item = seed['item_id']
            
            # 최적 배치를 찾았으므로 전체 대기열(job_queue)에서 제거 후 반환
            for job in best_batch:
                self.job_queue.remove(job)
                
            # [추가] ETA 로그 포함
            self.dispatch_log.append({
                'time': ctu.env.now,
                'ctu_id': ctu.id,
                'rule': 'load_balance',
                'seed_item': best_seed_item,
                'batch_items': [j['item_id'] for j in best_batch],
                'batch_size': len(best_batch),
                'eta': min_eta,
            })
            return best_batch


# ==========================================
# Slotting Rule
# ==========================================
def generate_fixed_inventory(rule_type, part_list, full_plan_df):
    inventory = {} # 부품 위치 저장용 딕셔너리 생성
    all_parts = sorted(part_list) # 모든 부품 코드 정렬
    
    # [수정] 랜덤 고정 시드 적용 -> 결과 재현
    random.seed(1234)
    np.random.seed(1234)
    
    # 공통: 부품별 출고 빈도 계산
    freq = full_plan_df['부품코드'].value_counts().to_dict()
    sorted_by_freq = sorted(all_parts, key=lambda x: freq.get(x, 0), reverse=True)
    n = len(all_parts)

    # 1. Random Slotting
    if rule_type == 'random':
        for part in all_parts:
            # 통로(X), 랙(Y), 선반(Z) 모두 random 선택
            inventory[part] = [random.choice(AISLE_X), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]

    # 2. ABC Slotting (수정됨: A02, A05 앞쪽 집중 배치)
    elif rule_type == 'abc':
        # 상위 20% A, 다음 30% B, 나머지 C
        nA = max(1, int(n * 0.2))
        nB = max(1, int(n * 0.3))
        A_parts = sorted_by_freq[:nA]
        B_parts = sorted_by_freq[nA:nA+nB]
        C_parts = sorted_by_freq[nA+nB:]

        # [수정] 통로(Aisle)를 의도에 맞게 명확히 분리
        a_aisles = [AISLE_X[1], AISLE_X[4]]  # A02(3.6), A05(10.8) - 스테이션 S01, S02와 가까운 통로
        b_aisles = [AISLE_X[2], AISLE_X[3]]  # A03(6.0), A04(8.4)  - 중간 통로
        c_aisles = [AISLE_X[0], AISLE_X[5]]  # A01(1.2), A06(13.2) - 외곽 통로
        
        def place_group(parts, aisles, y_range, z_indices):
            for i, part in enumerate(parts):
                aisle = aisles[i % len(aisles)]
                y = random.uniform(*y_range)
                z_idx = z_indices[i % len(z_indices)]
                z = SHELF_HEIGHTS[z_idx]
                inventory[part] = [aisle, round(y, 2), z]

        # A: A02, A05 통로 / 입구쪽(앞쪽) 집중 (Y: 4.0 ~ 8.0) / 하층 (Z: 0.0 ~ 1.0)
        place_group(A_parts, a_aisles, (RACK_START_Y, 8.0), [0, 1])
        
        # B: A03, A04 통로 / 중간 위치 (Y: 6.0 ~ 12.0) / 중간층 (Z: 1.0 ~ 2.0)
        place_group(B_parts, b_aisles, (6.0, 12.0), [2, 3])
        
        # C: A01, A06 통로 / 깊은 위치 (Y: 10.0 ~ 15.1) / 상층 (Z: 2.0 ~ 3.0)
        place_group(C_parts, c_aisles, (10.0, RACK_END_Y), [4, 5, 6])

    # 3. Characteristic Slotting
    elif rule_type == 'characteristic':
        group_u = [p for p in all_parts if str(p).startswith('U')]
        group_c = [p for p in all_parts if str(p).startswith('C')]
        others = [p for p in all_parts if p not in group_u + group_c]

        def place_group(parts, aisles):
            for i, part in enumerate(parts):
                aisle = aisles[i % len(aisles)]
                y = round(random.uniform(RACK_START_Y, RACK_END_Y), 2)
                z = random.choice(SHELF_HEIGHTS)
                inventory[part] = [aisle, y, z]

        # U는 좌측 Aisle, C는 우측 Aisle, 나머지는 가운데
        left_aisles = AISLE_X[:3]      # A01~A03
        right_aisles = AISLE_X[3:]     # A04~A06
        mid_aisles = AISLE_X[1:5]      # A02~A05
        
        place_group(group_u, left_aisles)
        place_group(group_c, right_aisles)
        place_group(others, mid_aisles)

    # 4. Kitting Slotting (같이 나가는 건 같이 둠)
    elif rule_type == 'kitting':
        # 주문 번호(No.)별로 묶어서 같이 주문된 부품 리스트를 추출
        order_groups = full_plan_df.groupby('No.')['부품코드'].apply(list)
        pairs = []
        for items in order_groups:
            # 2개 이상 주문된 건에 대해 모든 쌍(Pair)을 구함
            items = list(set(items))
            if len(items) > 1:
                pairs.extend(list(combinations(sorted(items), 2)))
        
        # 부품쌍의 등장 빈도 계산(많이 같이 등장한 조합 찾기)
        pair_counts = Counter(pairs)
        
        # 연관성 높은 부품부터 순서대로 리스트 생성
        related_parts = []
        for (p1, p2), count in pair_counts.most_common():
            if p1 in all_parts and p1 not in related_parts: 
                related_parts.append(p1)
            if p2 in all_parts and p2 not in related_parts: 
                related_parts.append(p2)
        # 연관성 데이터가 없는 나머지 부품들도 뒤에 추가
        for p in all_parts:
            if p not in related_parts: 
                related_parts.append(p)

        # 연관된 것끼리 같은 통로에 모으는 로직
        # 통로별 배치 수 계산
        parts_per_aisle = len(all_parts) // len(AISLE_X) + 1
        # 위치 배치
        for i, part in enumerate(related_parts):
            aisle_idx = i // parts_per_aisle
            aisle = AISLE_X[min(aisle_idx, len(AISLE_X)-1)]
            
            # 같은 통로 내에서도 인접하게 배치 (y 좌표 1m 이내)
            y_ratio = (i % parts_per_aisle) / parts_per_aisle
            y_pos = RACK_START_Y + y_ratio * (RACK_END_Y - RACK_START_Y)
            
            inventory[part] = [aisle, round(y_pos, 2), random.choice(SHELF_HEIGHTS)]
    
    return inventory


# ==========================================
# 하루 단위 시뮬레이션 실행
# 특정 날짜, 특정 규칙을 적용하여 하루치 작업을 시뮬레이션
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan):
    env = simpy.Environment() # 시뮬레이션 환경 생성
    warehouse = Warehouse(env, inventory, date_plan) # 창고 객체 생성
    
    # CTU 2대 생성 (각각 독립적인 프로세스로 동작 시작)
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)]
    
    # Generator
    def job_gen(env, wh):
        # 출고계획 데이터(plan_df)를 주문번호(No.)별로 그룹화
        for _, group in wh.plan_df.groupby('No.'):
            # 주문과 주문 사이에 2~5초 랜덤 간격 발생 (주문 유입 시뮬레이션)
            yield env.timeout(random.uniform(2, 5))
            
            # 해당 주문에 포함된 모든 부품을 큐에 추가
            for _, row in group.iterrows():
                p_id = row['부품코드']
                if p_id in wh.inventory:
                    # 큐에 작업 추가 (부품ID, 위치, 발생시간)
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now})

    # 종료 조건 확인용 모니터링 함수
    def monitor(env, warehouse, gen_proc, ctus):
        # 1. 모든 주문이 생성될 때까지 대기
        yield gen_proc
        
        # 2. 주문 생성이 끝났어도, 큐에 잔여 작업이 있거나 하나의 CTU라도 작업 중(busy=True)이면 반복 실행
        while warehouse.job_queue or any(c.busy for c in ctus):
            yield env.timeout(1) # 1초 단위로 시스템 상태를 계속 확인하도록 시간 진행

    # 주문 발생 프로세스 실행
    gen_proc = env.process(job_gen(env, warehouse))

    # 시뮬레이션 실행 시작
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus)))
    
    # 시뮬레이션 종료 후 결과 반환
    return {
        'Date': date, 
        'Time': round(env.now, 1), 
        'Distance': round(sum(c.total_distance for c in ctus), 1),
        'Move_log': pd.DataFrame(warehouse.move_log),
        'Dispatch_log': pd.DataFrame(warehouse.dispatch_log),
    }


# ==========================================
# [수정] 히트맵 시각화 함수 (전체 조합 표시용)
# ==========================================
def visualize_heatmap(move_df):
    """Aisle별 피킹 히트맵 시각화"""
    if move_df.empty:
        print("move_log 데이터가 없습니다.")
        return

    # 'RETURN' 이동을 제외하고, Y축에 표시할 텍스트 조합(Slotting × Dispatching) 컬럼 생성
    target_df = move_df[move_df['item_id'] != 'RETURN'].copy()
    target_df['Rule_Combo'] = target_df['Slotting'] + " × " + target_df['Dispatching']

    # Rule_Combo(행) x to_x(열) 기준으로 피킹 횟수 집계
    aisle_picks = target_df.groupby(['Rule_Combo', 'to_x']).size().unstack(fill_value=0)

    # 전체 조합(12개)을 넉넉히 보여주기 위해 세로 길이를 조정
    plt.figure(figsize=(12, 10))

    # Aisle 라벨을 보기 좋게 변환 (예: 첫 번째 열 → A01, 두 번째 열 → A02 ...)
    aisle_labels = [f"A{i+1:02d}" for i in range(len(aisle_picks.columns))]

    sns.heatmap(
        aisle_picks,  # 행이 조합, 열이 통로로 자동 매핑됨
        annot=True,
        fmt='d',
        cmap='YlOrRd',
        cbar_kws={'label': '피킹 횟수'},
        xticklabels=aisle_labels
    )

    plt.title('전체 조합 Aisle별 피킹 히트맵 (Slotting × Dispatching)', fontsize=16, fontweight='bold')
    plt.xlabel('Aisle (통로 번호)')
    plt.ylabel('알고리즘 조합 (Slotting × Dispatching)')
    plt.tight_layout()
    plt.show()


# ==========================================
# [추가] 로그 분석 함수
# ==========================================
def analyze_logs(move_df, dispatch_df, title_prefix=""):
    """로그 데이터 간단 분석"""
    print(f"\n{'='*60}")
    print(f" [{title_prefix}] 분석 결과")
    print(f"{'='*60}")
    
    if not move_df.empty:
        print(f"\n CTU별 총 이동거리:")
        ctu_dist = move_df.groupby('ctu_id')['travel_distance'].sum()
        print(ctu_dist.round(1))
        
        print(f"\n 평균 배치당 이동거리:")
        avg_dist_per_batch = move_df[move_df['item_id'] != 'RETURN'].groupby(
            ['dispatch_rule', 'ctu_id']
        )['travel_distance'].mean()
        print(avg_dist_per_batch.round(1))
    
    if not dispatch_df.empty:
        print(f"\n 배치 크기 통계:")
        batch_stats = dispatch_df.groupby('rule')['batch_size'].agg(['mean', 'std', 'min', 'max']).round(1)
        print(batch_stats)


# ==========================================
# main 함수
# ==========================================
def main():
    try:
        # outbound_plan.csv 파일을 읽어 전체 출고계획 데이터를 DataFrame으로 저장
        plan_df_full = pd.read_csv('outbound_plan.csv')
        # BOM 데이터 생성
        part_list = plan_df_full['부품코드'].unique()
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    # 출고계획 데이터에 존재하는 날짜 목록 추출 및 정렬
    unique_dates = sorted(plan_df_full['생산일자'].unique())
    
    # 적용한 Slotting 알고리즘 목록
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting'] 
    dispatch_rules = ['nearest', 'density', 'load_balance']     

    all_results_summary = []
    move_logs_all = []
    dispatch_logs_all = []

    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다.")
    
    # 모든 경우의 수(Grid Search) 실행
    # 1. Slotting rule 변경
    for s_rule in slotting_rules:
        fixed_inventory = generate_fixed_inventory(s_rule, part_list, plan_df_full)
        
        # 2. Dispatching rule 변경
        for d_rule in dispatch_rules:
            # 3. 날짜별 시뮬레이션 실행
            for date in unique_dates:
                # 해당 날짜 데이터만 필터링
                date_plan = plan_df_full[plan_df_full['생산일자'] == date].copy()
                
                # 시뮬레이션 1회 실행
                res = run_simulation(date, fixed_inventory, d_rule, date_plan)
                
                # 결과 딕셔너리에 사용된 Slotting + Dispatching Rule 정보 추가
                all_results_summary.append({
                    'Date': res['Date'],
                    'Time': res['Time'],
                    'Distance': res['Distance'],
                    'Slotting': s_rule,
                    'Dispatching': d_rule,
                })
                
                # [추가] 로그에 룰/날짜 정보 추가
                move_tmp = res['Move_log'].copy()
                move_tmp['Slotting'] = s_rule
                move_tmp['Dispatching'] = d_rule
                move_tmp['Date'] = date
                move_logs_all.append(move_tmp)
                
                disp_tmp = res['Dispatch_log'].copy()
                disp_tmp['Slotting'] = s_rule
                disp_tmp['Dispatching'] = d_rule
                disp_tmp['Date'] = date
                dispatch_logs_all.append(disp_tmp)

    # 결과 분석 및 출력
    df_res = pd.DataFrame(all_results_summary) # 모든 결과를 DataFrame 형태로 변환
    
    # 피벗 테이블 생성: 행 = Rule 조합, 열 = 날짜, 값 = 소요시간
    pivoted_time = df_res.pivot_table(index=['Slotting', 'Dispatching'], columns='Date', values='Time')
    
    # Rule 조합의 날짜별 총 소요시간 합계 계산
    pivoted_time['Total'] = pivoted_time.sum(axis=1)
    
    # 시간이 적게 걸린 순서로 정렬
    pivoted_time = pivoted_time.sort_values(by='Total')

    # =========================
    # --- 결과 표 & 그래프 ---
    # =========================
    print("\n" + "="*95)
    print(" [최종 시뮬레이션 결과] 알고리즘 조합별 일자별 런타임 요약")
    print("-" * 95)
    
    try:
        from IPython.display import display
        styled = pivoted_time.style.background_gradient(cmap='YlGnBu', subset=pivoted_time.columns[:-1]) \
            .highlight_min(subset=['Total'], color='lightgreen').format("{:.1f}")
        display(styled)
    except:
        print(pivoted_time.to_string())  # 텍스트 출력 fallback
        
    print("="*95)
    
    # 최적 알고리즘 조합 추출 및 출력
    best = pivoted_time.index[0]
    print(f" 최적 알고리즘 조합: Slotting[{best[0]}] + Dispatching[{best[1]}]")
    print(f"  최적 총 소요시간: {pivoted_time.loc[best, 'Total']:.1f}초")

    # 전체 로그 병합
    move_all_df = pd.concat(move_logs_all, ignore_index=True) if move_logs_all else pd.DataFrame()
    dispatch_all_df = pd.concat(dispatch_logs_all, ignore_index=True) if dispatch_logs_all else pd.DataFrame()
    
    # 로그 분석 출력
    analyze_logs(move_all_df, dispatch_all_df, title_prefix="전체 결과")
    
    # [수정] 히트맵 시각화 (전체 조합 모두 표시)
    print("\n 전체 알고리즘 조합에 대한 Aisle별 피킹 히트맵을 생성합니다...")
    visualize_heatmap(move_all_df)


# 현재 파일이 직접 실행될 때만 main() 함수 호출
if __name__ == "__main__":
    main()

In [ ]:
import simpy
import pandas as pd
import numpy as np
import random
from itertools import combinations  # 주문 내에서 함께 출고되는 부품쌍 생성 (Kitting Slotting에 사용)
from collections import Counter     # 함께 출고되는 부품쌍의 출현 빈도 계산 (어떤 부품을 서로 가깝게 배치할지 판단하기 위함)
import matplotlib.pyplot as plt     # 시각화 및 스타일링을 위한 라이브러리
import seaborn as sns               # 히트맵용 시각화 라이브러리

# 한글 폰트 설정 (히트맵 깨짐 해결)
plt.rcParams['font.family'] = 'Malgun Gothic'  # 윈도우 한글 폰트 (Mac의 경우 'AppleGothic' 등으로 변경)
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

# --- 시스템 환경 설정 (상수 정의) ---
# 창고의 물리적 레이아웃과 물류 로봇 성능 정의
RACK_START_Y, RACK_END_Y = 4.0, 15.1       # 랙이 설치된 Y축의 시작과 끝 좌표 범위 (4.0m ~ 15.1m 사이에만 적재 가능)
AISLE_X = [1.2, 3.6, 6.0, 8.4, 10.8, 13.2] # 창고 통로(Aisle)가 위치한 X축 좌표 리스트
STATIONS = {'S01': [3.5, 0.5, 0], 'S02': [11.0, 0.5, 0]} # 작업 스테이션 위치 (x, y, z), CTU가 작업 후 반드시 돌아오는 목적지
CTU_SPEED_XY = 1.8  # 로봇의 XY 이동 속도 (m/s)
CTU_SPEED_Z = 0.5   # 로봇의 리프트(Z) 상하 이동 속도 (m/s)
TOTE_CAPACITY = 7   # CTU가 한 번에 운반할 수 있는 최대 토트 박스 수
SHELF_HEIGHTS = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0] # 선반의 층별 높이(Z축) 리스트 (부품의 Z좌표로 사용)

# [수정] 교차 통로(Cross Aisle) 설정
# 로봇이 다른 통로로 넘어갈 때 랙을 관통하지 않고 앞쪽으로 빠져나와서 이동하는 기준 Y 좌표
CROSS_AISLE_Y = 2.0 

# ==========================================
# [신규] 창고 레이아웃 기반 맨해튼 직교 거리 계산 함수
# ==========================================
def get_manhattan_distance(pos1, pos2):
    """
    물류 로봇이 랙(Rack)을 뚫고 지나갈 수 없다는 물리적 제약을 반영한 거리 계산 함수.
    다른 통로로 이동할 때는 반드시 교차 통로(Y=2.0)로 나와서 이동해야 함.
    """
    x1, y1, z1 = pos1
    x2, y2, z2 = pos2
    
    # 1. 같은 통로(X좌표 동일)일 경우: 단순히 Y축 깊이 차이만 계산
    if x1 == x2:
        xy_dist = abs(y1 - y2)
    # 2. 다른 통로일 경우: 현재 통로에서 교차 통로(Y=2.0)로 직진하여 빠져나온 뒤, 가로(X)로 이동 후 새 통로로 진입
    else:
        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
        
    # Z축(높이) 차이 포함하여 최종 3D 맨해튼 거리 반환 (의사결정 우선순위용)
    return xy_dist + abs(z1 - z2)

# ==========================================
# CTU class 물류 로봇
# ==========================================
class CTU: # 물류 로봇 정의
    def __init__(self, env, id, warehouse, dispatch_rule): # env: 시뮬레이션 환경, id: CTU 번호, warehouse: 작업정보 접근, dispatch_rule: 작업 할당 알고리즘)
        self.env = env                  # Simpy 시뮬레이션 환경 객체 (현재 시뮬레이션 시간을 알고 있음)
        self.id = id                    # 로봇의 고유 번호 (0, 1, 2...)
        self.warehouse = warehouse      # 로봇이 소속된 창고 객체
        self.dispatch_rule = dispatch_rule # 이 로봇에게 적용될 작업 할당 알고리즘 (nearest, density, load_balance)
        
        # 초기 로봇 위치 설정: 0번 CTU는 S01에서, 1번 CTU는 S02에서 시작
        self.pos = STATIONS['S01'] if id == 0 else STATIONS['S02'] 

        # 총 이동거리 및 작업 수 기록
        self.total_distance = 0         # 시뮬레이션 동안 로봇이 이동한 총 누적 거리
        self.completed_jobs = 0         # 처리한 총 작업 수
        self.busy = False               # 로봇이 현재 작업 중인지 여부 (True: 작업중, False: 대기중)
        
        # 핵심: CTU의 작업 수행 프로세스를 시뮬레이션에 등록 (시간 흐름에 따라 run() 동작)
        self.process = env.process(self.run())

    # 이동 시간 및 거리 계산 (맨해튼 직교 거리 기반)
    def calculate_travel_time(self, target_pos):
        x1, y1, z1 = self.pos
        x2, y2, z2 = target_pos
        
        # 랙 관통 방지: 통로가 다르면 교차 통로(CROSS_AISLE_Y)를 경유
        if x1 == x2:
            xy_dist = abs(y1 - y2)
        else:
            xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
            
        dz = abs(z1 - z2)
        
        # 이동 시간 계산: XY 평면 이동과 Z축(리프트) 상하 이동을 동시에 수행한다고 가정
        travel_time = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
        
        # 총 이동 거리: 맨해튼 궤적 기반 실제 주행 거리 산출
        travel_distance = xy_dist + dz
        
        return travel_time, travel_distance

    # CTU의 실제 작업 수행 프로세스
    def run(self): 
        while True: # 시뮬레이션이 끝날 때까지 계속 반복
            # 1. 창고에 처리해야 할 일(job_queue)이 있는지 확인
            if self.warehouse.job_queue:
                self.busy = True # 작업 시작 상태로 변경
                
                # 2. 작업 할당 규칙에 따라 warehouse에 작업 묶음(batch)을 요청
                batch = self.warehouse.apply_dispatching(self, self.dispatch_rule)
                
                if batch:
                    # 3. 경로 최적화 (Greedy TSP)
                    # [수정] 직선거리가 아닌 맨해튼 직교 거리를 기준으로 가장 가까운 작업부터 방문하도록 정렬
                    batch.sort(key=lambda j: get_manhattan_distance(self.pos, j['pos']))
                    
                    # 4. 각 상품 위치로 이동하며 피킹 수행
                    for job in batch:
                        t, d = self.calculate_travel_time(job['pos'])
                        
                        # [추가] 이동 로그 기록
                        self.warehouse.move_log.append({
                            'time': self.env.now,
                            'ctu_id': self.id,
                            'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2],
                            'to_x': job['pos'][0], 'to_y': job['pos'][1], 'to_z': job['pos'][2],
                            'item_id': job['item_id'],
                            'travel_time': t, 'travel_distance': d,
                            'dispatch_rule': self.dispatch_rule,
                        })
                        
                        # 실제 로봇이 이동하는 시간(t초)만큼 시뮬레이션 시간을 대기시킴
                        yield self.env.timeout(t)
                        
                        self.pos = job['pos']     # 로봇 위치를 목표 지점으로 업데이트
                        self.total_distance += d  # 이동 거리 누적
                        self.completed_jobs += 1
                        
                    # 5. 모든 상품 피킹 완료 후, 가까운 스테이션으로 복귀
                    # [수정] 마지막 피킹 위치의 X좌표에 따라 복귀할 지정 스테이션 결정
                    last_job_x = self.pos[0]
                    # X좌표가 6.0 이하(A01, A02, A03)면 S01, 아니면(A04, A05, A06) S02로 고정 복귀
                    target_station = STATIONS['S01'] if last_job_x <= 6.0 else STATIONS['S02']
                    
                    t, d = self.calculate_travel_time(target_station)
                    
                    # 복귀 이동 로그 기록
                    self.warehouse.move_log.append({
                        'time': self.env.now,
                        'ctu_id': self.id,
                        'from_x': self.pos[0], 'from_y': self.pos[1], 'from_z': self.pos[2],
                        'to_x': target_station[0], 'to_y': target_station[1], 'to_z': target_station[2],
                        'item_id': 'RETURN',
                        'travel_time': t, 'travel_distance': d,
                        'dispatch_rule': self.dispatch_rule,
                    })
                    
                    yield self.env.timeout(t)
                    self.pos = target_station           
                    self.total_distance += d  
                
                self.busy = False # 작업 완료 후 상태를 대기중으로 변경
                
            else:
                # 6. 할 일이 없으면 1초 동안 대기했다가 다시 큐를 확인 (Busy Waiting 방지)
                yield self.env.timeout(1)


# ==========================================
# 창고 시스템 클래스 정의
# 재고 위치 및 작업 큐 관리 역할
# ==========================================
class Warehouse:
    def __init__(self, env, inventory, plan_df):
        self.env = env              # 시뮬레이션 환경 저장
        self.inventory = inventory  # {부품코드: [x, y, z]} 형태의 딕셔너리 (부품별 저장 위치 정보)
        self.plan_df = plan_df      # 해당 날짜에 처리해야 할 출고 계획 데이터프레임
        self.job_queue = []         # 실시간으로 들어오는 주문들이 쌓이는 대기열
        
        # [추가] 분석용 로그 리스트
        self.dispatch_log = []      # 배치 구성 로그
        self.move_log = []          # CTU 이동 로그

    # dispatching rule에 따라 CTU에게 할당할 작업 배치(최대 7개) 생성 
    def apply_dispatching(self, ctu, rule):
        if not self.job_queue: return []

        # ==========================================
        # 1. Nearest-seeded batch
        # ==========================================
        if rule == 'nearest':
            # [수정] 현재 위치에서 맨해튼 거리가 가장 가까운 작업을 Seed로 선택
            self.job_queue.sort(key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            batch = [self.job_queue.pop(0)] 
            
            if self.job_queue:
                # Seed 주변에서 맨해튼 거리가 가장 가까운 job들을 순서대로 담음
                self.job_queue.sort(key=lambda j: get_manhattan_distance(batch[0]['pos'], j['pos']))
                extra_count = min(len(self.job_queue), TOTE_CAPACITY - 1) # 한 번에 최대 7개 (TOTE_CAPACITY)
                batch.extend(self.job_queue[:extra_count])
                del self.job_queue[:extra_count]
                
            self.dispatch_log.append({
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'nearest',
                'seed_item': batch[0]['item_id'], 'batch_items': [j['item_id'] for j in batch],
                'batch_size': len(batch),
            })
            return batch

        # ==========================================
        # 2. Density-seeded batch 
        # ==========================================
        elif rule == 'density':
            # [수정] 맨해튼 거리 기준 가까운 10개의 작업을 후보 Seed로 추출
            K = min(10, len(self.job_queue)) 
            candidate_seeds = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))[:K]
            
            best_seed = None
            max_density = -1
            r_y = 2.65 # 반경 (1개 베이 길이 2.65m 적용)
            
            # 후보 Seed 각각에 대해 주변 밀집도 계산
            for seed in candidate_seeds:
                density = 0
                for job in self.job_queue:
                    # 동일 통로에 있으면서, Y축 반경 r_y 이내에 있는 대기 작업 수를 셈
                    if job['pos'][0] == seed['pos'][0]:
                        dy = abs(job['pos'][1] - seed['pos'][1])
                        if dy <= r_y:
                            density += 1
                
                if density > max_density:
                    max_density = density
                    best_seed = seed
                    
            # 가장 밀집도가 큰 지역의 Seed를 기준으로 가까운 작업들로 Batch 구성
            self.job_queue.sort(key=lambda j: get_manhattan_distance(best_seed['pos'], j['pos']))
            
            batch_size = min(len(self.job_queue), TOTE_CAPACITY)
            best_batch = self.job_queue[:batch_size]
            del self.job_queue[:batch_size]
            
            self.dispatch_log.append({
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'density',
                'seed_item': best_seed['item_id'], 'batch_items': [j['item_id'] for j in best_batch],
                'batch_size': len(best_batch), 'local_density': max_density,
            })
            return best_batch

        # ==========================================
        # 3. Load-balance batch (최단 예상 완료 시간 배정)
        # ==========================================
        elif rule == 'load_balance':
            # [수정] 맨해튼 거리 기준 상위 10개 작업 추출
            K = min(10, len(self.job_queue)) 
            nearest_jobs = sorted(self.job_queue, key=lambda j: get_manhattan_distance(ctu.pos, j['pos']))
            candidate_seeds = nearest_jobs[:K]
            
            best_batch = []
            min_eta = float('inf') 
            best_seed_item = None
            
            for seed in candidate_seeds:
                # 가상으로 배치 묶음을 만들어봄
                temp_queue = sorted(self.job_queue, key=lambda j: get_manhattan_distance(seed['pos'], j['pos']))
                temp_batch = temp_queue[:TOTE_CAPACITY]
                
                eta = 0 # 예상 처리 시간(Estimated Time of Arrival)
                current_pos = ctu.pos 
                
                # 해당 배치(임시)를 처리할 때 걸리는 총 소요시간 계산 (교차 통로 제약 포함)
                for job in temp_batch:
                    x1, y1, z1 = current_pos
                    x2, y2, z2 = job['pos']
                    
                    if x1 == x2:
                        xy_dist = abs(y1 - y2)
                    else:
                        xy_dist = abs(y1 - CROSS_AISLE_Y) + abs(x1 - x2) + abs(y2 - CROSS_AISLE_Y)
                        
                    dz = abs(z1 - z2)
                    time_to_job = max(xy_dist / CTU_SPEED_XY, dz / CTU_SPEED_Z)
                    eta += time_to_job
                    current_pos = job['pos'] 
                
                # 가장 빨리 끝나는(ETA가 최소인) 최적의 배치를 선택
                if eta < min_eta:
                    min_eta = eta
                    best_batch = temp_batch
                    best_seed_item = seed['item_id']
            
            for job in best_batch:
                self.job_queue.remove(job)
                
            self.dispatch_log.append({
                'time': ctu.env.now, 'ctu_id': ctu.id, 'rule': 'load_balance',
                'seed_item': best_seed_item, 'batch_items': [j['item_id'] for j in best_batch],
                'batch_size': len(best_batch), 'eta': min_eta,
            })
            return best_batch


# ==========================================
# Slotting Rule (하이브리드: ABC 앞쪽 집중 전략 적용)
# ==========================================
def generate_fixed_inventory(rule_type, part_list, full_plan_df):
    inventory = {} 
    all_parts = sorted(part_list) # 모든 부품 코드 정렬
    
    # 랜덤 고정 시드 적용 -> 날짜별 시뮬레이션 결과 재현 가능
    random.seed(1234)
    np.random.seed(1234)
    
    # 공통: 부품별 출고 빈도 계산 (가장 많이 주문되는 부품 찾기 위함)
    freq = full_plan_df['부품코드'].value_counts().to_dict()
    sorted_by_freq = sorted(all_parts, key=lambda x: freq.get(x, 0), reverse=True)
    n = len(all_parts)

    # 1. Random Slotting
    if rule_type == 'random':
        for part in all_parts:
            # 통로(X), 랙(Y), 선반(Z) 위치를 무작위로 할당
            inventory[part] = [random.choice(AISLE_X), round(random.uniform(RACK_START_Y, RACK_END_Y), 2), random.choice(SHELF_HEIGHTS)]

    # ==========================================
    # 2. ABC Slotting (수정됨: 골든 존 집중 배치)
    # ==========================================
    elif rule_type == 'abc':
        # 상위 20%를 A등급, 다음 30%를 B등급, 나머지 50%를 C등급으로 분류
        nA = max(1, int(n * 0.2))
        nB = max(1, int(n * 0.3))
        A_parts = sorted_by_freq[:nA]
        B_parts = sorted_by_freq[nA:nA+nB]
        C_parts = sorted_by_freq[nA+nB:]

        # [수정] 통로 역할을 명확히 분리: 스테이션과 가장 가까운 A02, A05를 최우선 통로로 지정
        a_aisles = [AISLE_X[1], AISLE_X[4]]  # A02(3.6), A05(10.8) 
        b_aisles = [AISLE_X[2], AISLE_X[3]]  # A03(6.0), A04(8.4)  
        c_aisles = [AISLE_X[0], AISLE_X[5]]  # A01(1.2), A06(13.2) 
        
        def place_group(parts, aisles, y_range, z_indices):
            for i, part in enumerate(parts):
                aisle = aisles[i % len(aisles)]
                y = random.uniform(*y_range)
                z_idx = z_indices[i % len(z_indices)]
                z = SHELF_HEIGHTS[z_idx]
                inventory[part] = [aisle, round(y, 2), z]

        # [수정] A등급: 스테이션 근처(A02, A05) + 랙 맨 앞쪽(Y: 4~8m) + 하층부(1~2층) 골든존에 몰아넣음
        place_group(A_parts, a_aisles, (RACK_START_Y, 8.0), [0, 1])
        # [수정] B등급: 중간 통로(A03, A04) + 중간 위치(Y: 6~12m) + 중간층(3~4층)
        place_group(B_parts, b_aisles, (6.0, 12.0), [2, 3])
        # [수정] C등급: 외곽 통로(A01, A06) + 구석 위치(Y: 10~15.1m) + 상층부(5~7층)
        place_group(C_parts, c_aisles, (10.0, RACK_END_Y), [4, 5, 6])

    # ==========================================
    # 3. Characteristic Slotting (신규: 특성별 분류 + ABC 하이브리드)
    # ==========================================
    elif rule_type == 'characteristic':
        # [수정] U나 C 특성으로 그룹을 나누되, 그 안에서도 '출고 빈도'가 높은 순으로 정렬함
        group_u = sorted([p for p in all_parts if str(p).startswith('U')], key=lambda x: freq.get(x, 0), reverse=True)
        group_c = sorted([p for p in all_parts if str(p).startswith('C')], key=lambda x: freq.get(x, 0), reverse=True)
        others = sorted([p for p in all_parts if p not in group_u + group_c], key=lambda x: freq.get(x, 0), reverse=True)

        def place_by_abc_tier(parts, tiers):
            if not parts: return
            n = len(parts)
            nA, nB = max(1, int(n * 0.2)), max(1, int(n * 0.3)) # 각 특성 그룹 내에서도 20%, 30%, 50% 분할
            
            chunks = [parts[:nA], parts[nA:nA+nB], parts[nA+nB:]]
            
            # 잘 나가는 부품일수록 좋은 티어(A02/A05 앞쪽)의 위치 정보를 매핑
            for idx, chunk in enumerate(chunks):
                tier = tiers[idx]
                for i, part in enumerate(chunk):
                    aisle = tier['aisles'][i % len(tier['aisles'])]
                    y = random.uniform(*tier['y_range'])
                    z = SHELF_HEIGHTS[tier['z_indices'][i % len(tier['z_indices'])]]
                    inventory[part] = [aisle, round(y, 2), z]

        # [수정] U그룹: 왼쪽 통로 할당, A등급은 A02 앞쪽 차지
        u_tiers = [
            {'aisles': [AISLE_X[1]], 'y_range': (RACK_START_Y, 8.0), 'z_indices': [0, 1]},   
            {'aisles': [AISLE_X[2]], 'y_range': (6.0, 12.0), 'z_indices': [2, 3]},           
            {'aisles': [AISLE_X[0]], 'y_range': (10.0, RACK_END_Y), 'z_indices': [4, 5, 6]}  
        ]
        
        # [수정] C그룹: 오른쪽 통로 할당, A등급은 A05 앞쪽 차지
        c_tiers = [
            {'aisles': [AISLE_X[4]], 'y_range': (RACK_START_Y, 8.0), 'z_indices': [0, 1]},   
            {'aisles': [AISLE_X[3]], 'y_range': (6.0, 12.0), 'z_indices': [2, 3]},           
            {'aisles': [AISLE_X[5]], 'y_range': (10.0, RACK_END_Y), 'z_indices': [4, 5, 6]}  
        ]
        
        # 나머지 부품
        o_tiers = [
            {'aisles': [AISLE_X[1], AISLE_X[4]], 'y_range': (RACK_START_Y, 8.0), 'z_indices': [0, 1]},
            {'aisles': [AISLE_X[2], AISLE_X[3]], 'y_range': (6.0, 12.0), 'z_indices': [2, 3]},
            {'aisles': [AISLE_X[0], AISLE_X[5]], 'y_range': (10.0, RACK_END_Y), 'z_indices': [4, 5, 6]}
        ]

        place_by_abc_tier(group_u, u_tiers)
        place_by_abc_tier(group_c, c_tiers)
        place_by_abc_tier(others, o_tiers)

    # ==========================================
    # 4. Kitting Slotting (신규: 동시 출고 빈도순 정렬 + 우선순위 통로 적용)
    # ==========================================
    elif rule_type == 'kitting':
        # 주문 번호별로 같이 묶여 나가는 부품쌍을 모두 구함
        order_groups = full_plan_df.groupby('No.')['부품코드'].apply(list)
        pairs = []
        for items in order_groups:
            items = list(set(items))
            if len(items) > 1:
                pairs.extend(list(combinations(sorted(items), 2)))
        
        # [수정] 부품쌍의 등장 빈도 계산 (가장 자주 같이 나가는 '베스트셀러 세트'를 찾음)
        pair_counts = Counter(pairs)
        
        # 베스트셀러 세트부터 차례대로 리스트업
        related_parts = []
        for (p1, p2), count in pair_counts.most_common():
            if p1 in all_parts and p1 not in related_parts: related_parts.append(p1)
            if p2 in all_parts and p2 not in related_parts: related_parts.append(p2)
            
        for p in all_parts:
            if p not in related_parts: related_parts.append(p)

        # [수정] 통로에 무작위로 채우지 않고, 스테이션과 가장 가까운 A02/A05부터 우선적으로 꽉꽉 채움
        prioritized_aisles = [AISLE_X[1], AISLE_X[4], AISLE_X[2], AISLE_X[3], AISLE_X[0], AISLE_X[5]]
        parts_per_aisle = len(all_parts) // len(prioritized_aisles) + 1

        for i, part in enumerate(related_parts):
            aisle_idx = i // parts_per_aisle
            aisle = prioritized_aisles[min(aisle_idx, len(prioritized_aisles)-1)]
            
            # 같은 통로 내에서도 우선순위가 높은 부품일수록 앞쪽(y_ratio가 낮음)에 배치
            y_ratio = (i % parts_per_aisle) / parts_per_aisle
            y_pos = RACK_START_Y + y_ratio * (RACK_END_Y - RACK_START_Y)
            
            # [수정] 앞쪽에 배치되는 주요 키트일수록 Z축(높이)도 하층부에 위치시켜 로봇 승강 시간 단축
            if y_ratio < 0.3: z = random.choice(SHELF_HEIGHTS[0:2])
            elif y_ratio < 0.6: z = random.choice(SHELF_HEIGHTS[2:4])
            else: z = random.choice(SHELF_HEIGHTS[4:])
            
            inventory[part] = [aisle, round(y_pos, 2), z]
            
    return inventory


# ==========================================
# 하루 단위 시뮬레이션 실행
# ==========================================
def run_simulation(date, inventory, dispatch_rule, date_plan):
    env = simpy.Environment() # 시뮬레이션 환경 생성
    warehouse = Warehouse(env, inventory, date_plan) # 창고 객체 생성
    
    # CTU 2대 생성 (각각 독립적인 프로세스로 동작 시작)
    ctus = [CTU(env, i, warehouse, dispatch_rule) for i in range(2)]
    
    # 주문 발생 제너레이터: 출고 계획 데이터를 시간 간격을 두고 큐에 넣음
    def job_gen(env, wh):
        for _, group in wh.plan_df.groupby('No.'):
            yield env.timeout(random.uniform(2, 5)) # 주문 발생 사이의 랜덤 간격
            for _, row in group.iterrows():
                p_id = row['부품코드']
                if p_id in wh.inventory:
                    wh.job_queue.append({'item_id': p_id, 'pos': wh.inventory[p_id], 'time': env.now})

    # 종료 조건 모니터링: 모든 주문 생성이 끝나고, 큐가 비어있고, 로봇들이 대기 상태일 때 종료
    def monitor(env, warehouse, gen_proc, ctus):
        yield gen_proc
        while warehouse.job_queue or any(c.busy for c in ctus):
            yield env.timeout(1) 

    gen_proc = env.process(job_gen(env, warehouse))
    env.run(until=env.process(monitor(env, warehouse, gen_proc, ctus)))
    
    # 해당 날짜의 총 소요시간 및 이동 거리 반환
    return {
        'Date': date, 
        'Time': round(env.now, 1), 
        'Distance': round(sum(c.total_distance for c in ctus), 1),
        'Move_log': pd.DataFrame(warehouse.move_log),
        'Dispatch_log': pd.DataFrame(warehouse.dispatch_log),
    }


# ==========================================
# 히트맵 시각화 함수 (전체 조합 표시용)
# ==========================================
def visualize_heatmap(move_df):
    """
    12가지 알고리즘 조합(Slotting × Dispatching) 각각에 대해
    어느 통로(Aisle)에 방문이 집중되었는지 한눈에 비교하는 히트맵 시각화
    """
    if move_df.empty:
        print("move_log 데이터가 없습니다.")
        return

    # 스테이션 복귀(RETURN) 로그 제외
    target_df = move_df[move_df['item_id'] != 'RETURN'].copy()
    
    # [추가] Y축 표시를 위해 '슬로팅 × 디스패칭' 형태의 파생 변수(Rule_Combo) 생성
    target_df['Rule_Combo'] = target_df['Slotting'] + " × " + target_df['Dispatching']

    # X축은 통로(to_x), Y축은 알고리즘 조합(Rule_Combo)으로 피벗 집계
    aisle_picks = target_df.groupby(['Rule_Combo', 'to_x']).size().unstack(fill_value=0)

    # 12개의 행이 잘리지 않도록 세로 길이를 넉넉하게 10으로 설정
    plt.figure(figsize=(12, 10))

    # X축 통로 이름 포맷팅 (예: A01, A02)
    aisle_labels = [f"A{i+1:02d}" for i in range(len(aisle_picks.columns))]

    sns.heatmap(
        aisle_picks,  
        annot=True,
        fmt='d',
        cmap='YlOrRd',
        cbar_kws={'label': '피킹 횟수'},
        xticklabels=aisle_labels
    )

    plt.title('전체 알고리즘 조합 통로(Aisle)별 피킹 히트맵', fontsize=16, fontweight='bold')
    plt.xlabel('Aisle (통로 번호)')
    plt.ylabel('알고리즘 조합 (Slotting × Dispatching)')
    plt.tight_layout()
    plt.show()


# ==========================================
# 로그 분석 함수
# ==========================================
def analyze_logs(move_df, dispatch_df, title_prefix=""):
    """CTU별 이동 거리 및 배치 크기 통계 요약 출력"""
    print(f"\n{'='*60}")
    print(f" [{title_prefix}] 분석 결과")
    print(f"{'='*60}")
    
    if not move_df.empty:
        print(f"\n CTU별 총 이동거리:")
        ctu_dist = move_df.groupby('ctu_id')['travel_distance'].sum()
        print(ctu_dist.round(1))
        
        print(f"\n 평균 배치당 이동거리:")
        avg_dist_per_batch = move_df[move_df['item_id'] != 'RETURN'].groupby(
            ['dispatch_rule', 'ctu_id']
        )['travel_distance'].mean()
        print(avg_dist_per_batch.round(1))
    
    if not dispatch_df.empty:
        print(f"\n 배치 크기 통계:")
        batch_stats = dispatch_df.groupby('rule')['batch_size'].agg(['mean', 'std', 'min', 'max']).round(1)
        print(batch_stats)


# ==========================================
# main 함수
# ==========================================
def main():
    try:
        # 데이터 파일 로드 및 BOM(부품 목록) 추출
        plan_df_full = pd.read_csv('outbound_plan.csv')
        part_list = plan_df_full['부품코드'].unique()
    except Exception as e:
        print(f"파일을 읽는 중 오류가 발생했습니다: {e}")
        return

    unique_dates = sorted(plan_df_full['생산일자'].unique())
    
    # Grid Search를 위한 모든 알고리즘 경우의 수 정의
    slotting_rules = ['random', 'abc', 'characteristic', 'kitting'] 
    dispatch_rules = ['nearest', 'density', 'load_balance']     

    all_results_summary = []
    move_logs_all = []
    dispatch_logs_all = []

    print(f"총 {len(unique_dates)}일치 데이터에 대해 시뮬레이션을 시작합니다.")
    
    # 1. Slotting rule 변경 루프
    for s_rule in slotting_rules:
        # 동일한 룰 내에서는 같은 재고 위치를 유지
        fixed_inventory = generate_fixed_inventory(s_rule, part_list, plan_df_full)
        
        # 2. Dispatching rule 변경 루프
        for d_rule in dispatch_rules:
            # 3. 각 날짜별로 시뮬레이션 수행
            for date in unique_dates:
                date_plan = plan_df_full[plan_df_full['생산일자'] == date].copy()
                
                # 시뮬레이션 실행
                res = run_simulation(date, fixed_inventory, d_rule, date_plan)
                
                # 요약 결과 저장
                all_results_summary.append({
                    'Date': res['Date'],
                    'Time': res['Time'],
                    'Distance': res['Distance'],
                    'Slotting': s_rule,
                    'Dispatching': d_rule,
                })
                
                # 전체 이동 로그에 현재 수행한 룰과 날짜 정보 마킹 후 저장
                move_tmp = res['Move_log'].copy()
                move_tmp['Slotting'] = s_rule
                move_tmp['Dispatching'] = d_rule
                move_tmp['Date'] = date
                move_logs_all.append(move_tmp)
                
                # 전체 배치 할당 로그에 현재 수행한 룰과 날짜 정보 마킹 후 저장
                disp_tmp = res['Dispatch_log'].copy()
                disp_tmp['Slotting'] = s_rule
                disp_tmp['Dispatching'] = d_rule
                disp_tmp['Date'] = date
                dispatch_logs_all.append(disp_tmp)

    # 전체 요약 결과를 데이터프레임으로 변환
    df_res = pd.DataFrame(all_results_summary) 
    
    # 알고리즘 조합별(행) / 날짜별(열) 소요시간 피벗 테이블 생성
    pivoted_time = df_res.pivot_table(index=['Slotting', 'Dispatching'], columns='Date', values='Time')
    pivoted_time['Total'] = pivoted_time.sum(axis=1) # 총 소요 시간 합산
    pivoted_time = pivoted_time.sort_values(by='Total') # 가장 빠른 조합 순으로 정렬

    print("\n" + "="*95)
    print(" [최종 시뮬레이션 결과] 알고리즘 조합별 일자별 런타임 요약")
    print("-" * 95)
    
    try:
        from IPython.display import display
        styled = pivoted_time.style.background_gradient(cmap='YlGnBu', subset=pivoted_time.columns[:-1]) \
            .highlight_min(subset=['Total'], color='lightgreen').format("{:.1f}")
        display(styled)
    except:
        print(pivoted_time.to_string())  
        
    print("="*95)
    
    # 최우수 알고리즘 조합 출력
    best = pivoted_time.index[0]
    print(f" 최적 알고리즘 조합: Slotting[{best[0]}] + Dispatching[{best[1]}]")
    print(f"  최적 총 소요시간: {pivoted_time.loc[best, 'Total']:.1f}초")

    # 전체 로그 데이터프레임 병합
    move_all_df = pd.concat(move_logs_all, ignore_index=True) if move_logs_all else pd.DataFrame()
    dispatch_all_df = pd.concat(dispatch_logs_all, ignore_index=True) if dispatch_logs_all else pd.DataFrame()
    
    # 종합 로그 분석 결과 출력
    analyze_logs(move_all_df, dispatch_all_df, title_prefix="전체 결과")
    
    # 전체 조합을 한눈에 볼 수 있는 통합 히트맵 생성
    print("\n 전체 알고리즘 조합에 대한 Aisle별 피킹 히트맵을 생성합니다...")
    visualize_heatmap(move_all_df)


if __name__ == "__main__":
    main()